# Shot Quality in Rocket League 2v2
CPSC 322 Final Project Proposal By Tom and Vincent

## Project question

If we have the position of the ball at contact and the speed of the shot, how likely is an on target shot to score in Rocket League 2v2? We will test whether the buildup to the shot, score at the time, and the shooters boost amount improve the prediction beyond knowing the angle and speed. We both have spent a significant amount of time playing rocket league, in fact, we are both starters on Gonzagas Rocket League Team! We are always looking to get better and this project will let us test some of our judgements. Each shot has a binary outcome. You either miss it or you make it. We plan on answering whether the best models probabilites agree with the actual scoring rates. If they do, the estimated probability can serve as a post shot expected goals value that we can use to see if we are getting better.

## Dataset

In one of Toms past projects, he collected Rocket league stats for around 300 matches. The original data came from a personal Rocket League tracker that recorded Stats API events and player telemetry in SQLite. The export joins game-recorded shot events to ball contacts, preceding goal events, and, where available, player telemetry. This data is inside rocket_league_2v2.csv. It contains 3813 on target shots from 232 recorded 2v2 matches. Of those shots, 1237 scored (32.4%) and 2576 (67.6%) did not.
Here is a link to the api that was used to gather data: [Psyonix Rocket League Stats API documentation](https://www.rocketleague.com/developer/stats-api?lang=en)

### Attributes

| CSV column(s) | Type | Meaning |
| --- | --- | --- |
| `contact_x_uu`, `contact_y_uu`, `contact_z_uu` | Numeric | Ball position at the shooting touch, in Unreal Units and rotated so both teams attack toward positive Y. |
| `goal_distance_uu`, `goal_angle_rad` | Numeric | Distance from contact to the center of the opponent's goal and the horizontal angle which is subtended by the opening of the goal |
| `ball_speed_before_recorded`, `ball_speed_after_recorded` | Numeric | Recorded ball speeds immediately before and after the contact. |
| `previous_touch`, `previous_touch_gap_s` | Categorical, numeric | Who made the previous touch and how many seconds earlier it occurred, considering a five-second window. Simultaneous or unattributed touches are marked ambiguous. |
| `score_diff_before` | Integer | The shooting teams lead or deficit at contact, calculated from earlier goals. |
| `shooter_boost`, `shooter_boosting` | Numeric, binary | Boost in the tank (0–100) and whether the shooter was boosting, from the latest valid sample within 250 milliseconds before contact. |
| `match_id` | Integer identifier | Used to keep shots from the same match together during evaluation. |
| `goal` | Binary class label | 1 if the shot was matched to a goal credited to the shooter, and 0 otherwise. |


## Implementation / evaluation

- Preprocessing. Rapid sequences can make it difficult to associate a goal with the correct touch. We will examine the recorded speed and use the same unit convention everywhere we can. We also want to = restrict features to information available at or immediately before the shooting touch, including the balls immediate post-contact speed. Also Fields observed later, such as the location where a scored shot crossed the goal line, would probably reveal the outcome and will be excluded.

- Exploratory analysis and feature selection. We will plot everything! (shot locations and compare conversion rates by distance, angle, and previous-touch category). It should be noted that, Distance, angle, and the original coordinates contain overlapping information. Unfortunately boost is available for 774 shots across 98 matches. Because of this small amount of games with boost we will compare models with and without boost on the same covered shots and check how that subset differs from the full dataset. We will use the training data to select features, fit any scaling, and choose bin boundaries if our tree implementation requires discretization.

- Classifiers. We plan to compare a single entropy-based decision tree, k-nearest neighbors, and a custom MyRandomForestClassifier. We will write tests before implementing the forests bootstrap sampling, random feature selection at each split, tree selection by out-of-bag accuracy, and majority voting. As for settings, they will be compared over repeated training splits.

Evaluation. We will keep whole matches together and aim for a ~roughly stratified one-third test split by shot count. All three classifiers will use the same held-out matches. We will report accuracy, confusion matrices, and goal recall because predicting no goal every time would already achieve about 67.6% accuracy on the full sample. With regards to actual xG interpretation, we will also compare the forests vote fractions with observed scoring rates on held-out shots. Finally we will report how much performance changes across splits and whether any improvement from score or boost holds across matches

## Potential impact

Competitive 2v2 players, coaches, and any replay analysts could use the estimates to prioritize clips for review and discuss chance creation and finishing separately.For example, a team could compare how often a recent teammate touch leads to a stronger opportunity with how often it converts from similar locations. We also think it would be cool to see how well classifiers will handle the complex game of Rocket League. If boost or score adds little predictive value in the recorded matches, that finding would help identify which features belong in a simpler replay tool. The analysis will measure predictive associations in this sample. A boost association alone would leave open whether spending more boost would improve the outcome.

## Sources and acknowledgments

- Psyonix. [Rocket League Stats API documentation](https://www.rocketleague.com/developer/stats-api?lang=en). Source of the ball-contact, player, and match events captured by the tracker.
- Toms prior project, RL Stats Tracker. The local rlstats.db SQLite capture is the source of the CSV. 